# Gradients in PyTorch

## The problem this solves

To train a model, you need to know how much each weight contributed to the error, so you know which direction to adjust it (gradient descent). Computing these derivatives by hand for a network with millions of parameters would be unmanageable. PyTorch does this automatically through **autograd** — automatic differentiation.

---

## Is `requires_grad` True by default? — depends on what you're looking at

**Plain tensors created directly (`torch.tensor`, `torch.rand`, etc.) default to `False`.**

```python
x = torch.rand(3)
x.requires_grad   # → False
```

Tracking gradients has a memory/compute cost, so PyTorch doesn't turn it on unless you ask for it.

**Model parameters (`nn.Parameter`, created inside layers like `nn.Linear`) default to `True`.**

```python
layer = nn.Linear(5, 3)
layer.weight.requires_grad   # → True
layer.bias.requires_grad     # → True
```

This makes sense — the whole point of a model's parameters is to be trained, so they're tracked from the start. You'd manually set `requires_grad = False` on them if you wanted to freeze a layer (common in fine-tuning).

**Summary:**

| Tensor type | Default `requires_grad` |
|---|---|
| Plain tensor (`torch.tensor`, `torch.rand`, ...) | `False` |
| `nn.Parameter` / model weights (`nn.Linear`, etc.) | `True` |

---

## `requires_grad = True`

This is the flag that tells PyTorch: "track every operation done with this tensor, so a derivative can be computed later."

```python
x = torch.tensor(2.0, requires_grad=True)
```

Once set, any operation involving `x` gets recorded into an internal computation graph. This is what allows `.backward()` to later walk that graph backwards and compute derivatives via the chain rule.

**Propagation rule:** if you combine tensors and at least one of them has `requires_grad=True`, the result also becomes `requires_grad=True` automatically.

```python
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0)          # requires_grad=False

c = a + b
c.requires_grad                # → True (inherited from a)
```

**Checking if an operation was actually tracked:** look at `.grad_fn`. If it's not `None`, the tensor is connected to the computation graph.

```python
y = x ** 2
y.grad_fn   # → <PowBackward0 object at ...>  → confirms it's being tracked
```

**Turning tracking off temporarily** — common during evaluation/inference, where gradients aren't needed and tracking them just wastes memory:

```python
with torch.no_grad():
    y = x ** 2
    y.requires_grad   # → False, even though x.requires_grad is True
```

---

## `.grad`

This is the attribute where the computed gradient gets stored, **after** you call `.backward()`.

```python
x = torch.tensor(2.0, requires_grad=True)
print(x.grad)          # → None (nothing computed yet)

y = x ** 2
y.backward()            # computes dy/dx

print(x.grad)           # → tensor(4.0)   (since dy/dx = 2x = 2*2 = 4)
```

Key points about `.grad`:

- It starts as `None` and only gets a value after `.backward()` runs.
- It only exists (gets populated) for tensors that had `requires_grad=True`.
- Calling `.backward()` again **accumulates** into `.grad` instead of overwriting it — this is why the training loop needs to zero it out each iteration:

```python
for x_batch, y_batch in loader:
    pred = model(x_batch)
    loss = criterion(pred, y_batch)

    loss.backward()          # computes gradients, adds them to .grad
    optimizer.step()         # uses .grad to update weights
    optimizer.zero_grad()    # resets .grad to zero for the next batch
```

Without `zero_grad()`, gradients from previous batches would keep piling up on top of the new ones, corrupting the update direction.

---

## `.requires_grad`

This is the readable flag itself — a boolean attribute you check (or set) on any tensor.

```python
x.requires_grad          # check the current state → True / False
x.requires_grad = False  # turn tracking off (e.g. freezing a layer)
x.requires_grad_(True)   # in-place version (note the trailing underscore)
```

Common use case — freezing part of a model during fine-tuning:

```python
for param in model.backbone.parameters():
    param.requires_grad = False   # these layers won't be updated

for param in model.head.parameters():
    param.requires_grad = True    # only train the new head
```

Checking all parameters of a model at once:

```python
for name, param in model.named_parameters():
    print(name, param.requires_grad)
```

---

## Quick reference — how to tell if gradient tracking is actually happening

| What to check | How |
|---|---|
| Is the tensor marked to track gradients | `x.requires_grad` |
| Was an operation actually recorded into the graph | `x.grad_fn` (not `None` = tracked) |
| Has a gradient already been computed | `x.grad` (not `None` = `.backward()` already ran) |
| Is gradient tracking globally enabled right now | `torch.is_grad_enabled()` |
| Which model parameters are trainable | `for p in model.parameters(): print(p.requires_grad)` |

---

## Minimal end-to-end example

```python
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2          # y = x²

y.backward()         # computes dy/dx via the chain rule

print(x.grad)         # → tensor(4.0), since dy/dx = 2x = 4 when x=2
```

This is exactly the mechanism behind model training: `loss.backward()` computes the gradient of the loss with respect to every trainable parameter in the model, and `optimizer.step()` uses those gradients to update the weights.

In [5]:
import torch

tensor = torch.tensor(2, requires_grad=True, dtype=torch.float16)

print(f'Gradient on?: {tensor.requires_grad}')
print(f'Gradient without calculus: {tensor.grad}')

Gradient on?: True
Gradient without calculus: None


In [ ]:
y = tensor ** 2
y.backward()

print(f'Gradient after calcukus: {tensor.grad}')

Gradient after calcukus: 4.0


In [ ]:
import torch.nn as nn

v = torch.tensor(2)
w = nn.Linear(2, 2)
z = torch.rand(2, 2)

print(v.requires_grad)
print(w.weight.requires_grad)
print(z.requires_grad)

False
True
False


: 